In [63]:
# Finance set

tableName='finance_markers'
filterColumn='investigationState'
filterValue='Complete'
classifierColumn='fraudtype'

In [64]:
# Some data exploration

%load_ext sql
%sql trino://streambased-server:8080/kafka
%sql SELECT * FROM kafka.streambased.{{tableName}} LIMIT 10


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Running query in 'trino://streambased-server:8080/kafka'

fraudtype,transcountabove1stddev,multipleinternationtransactions,multipleaddresschanges,transamountabove1stddev,complaints,above10adjustments,multipledocumentverificationattempts,accountmorethan6monthsold,investigationstate,_partition_id,_partition_offset,_message_corrupt,_message,_headers,_message_length,_key_corrupt,_key,_key_length,_timestamp
Credit Card Fraud,1,0,0,1,1,1,1,1,Complete,0,0,False,"    ""Credit Card Fraud  Complete",{},40,False,None,0,2024-03-04 11:02:33.242000
Payment Fraud,1,1,1,0,0,1,1,0,Complete,0,1,False,    Payment Fraud   Complete,{},36,False,None,0,2024-03-04 11:02:33.254000
Credit Card Fraud,1,0,0,1,1,1,1,0,Complete,0,2,False,"    ""Credit Card Fraud   Complete",{},40,False,None,0,2024-03-04 11:02:33.254000
Insurance Fraud,0,1,0,1,1,0,1,0,Complete,0,3,False,     Insurance Fraud    Complete,{},38,False,None,0,2024-03-04 11:02:33.255000
Credit Card Fraud,0,1,1,1,0,0,1,0,Complete,0,4,False,"    ""Credit Card Fraud    Complete",{},40,False,None,0,2024-03-04 11:02:33.255000
Tax Evasion,1,0,0,1,1,1,0,0,Complete,0,5,False,    Tax Evasion    Complete,{},34,False,None,0,2024-03-04 11:02:33.255000
Credit Card Fraud,0,1,1,1,1,1,1,0,Complete,0,6,False,"    ""Credit Card Fraud  Complete",{},40,False,None,0,2024-03-04 11:02:33.255000
Insurance Fraud,1,1,0,1,0,0,1,0,Complete,0,7,False,     Insurance Fraud    Complete,{},38,False,None,0,2024-03-04 11:02:33.255000
Tax Evasion,0,0,1,1,1,0,1,1,Complete,0,8,False,    Tax Evasion   Complete,{},34,False,None,0,2024-03-04 11:02:33.256000
Payment Fraud,0,0,1,0,0,0,0,1,Complete,0,9,False,    Payment Fraud      Complete,{},36,False,None,0,2024-03-04 11:02:33.256000


In [65]:
# Fetch our relevant dataset

result = %sql SELECT * FROM kafka.streambased.{{tableName}} WHERE {{filterColumn}}='{{filterValue}}'
import pandas as pd
df = result.DataFrame()

Running query in 'trino://streambased-server:8080/kafka'

In [59]:
from river import tree, metrics
from joblib import dump, load

features = [column for column in df.columns if column not in [filterColumn, '_partition_id', '_partition_offset',
                                                               '_message_corrupt', '_message', '_headers',
                                                               '_message_length', '_key_corrupt', '_key',
                                                               '_key_length', '_timestamp', classifierColumn]]
#classifierColumn = 'fraudtype'   Replace with your actual target column name

# Initialize your model with example parameters
model = tree.HoeffdingTreeClassifier(
    grace_period=25,
    max_depth=30,
    split_criterion='gini',
    leaf_prediction='nba',
    nb_threshold=10,
    binary_split=True,
    min_branch_fraction=0.005,
    max_share_to_split=0.95,
    max_size=200
)

# Prepare a metric to track performance
metric = metrics.Accuracy()
total_trained_rows = 0
# Assuming 'df' is your DataFrame and 'features' is a list of feature column names
# Assuming 'classifierColumn' is the name of your target variable column
for index, row in df.iterrows():
    x = row[features].to_dict()
    y = row['fraudtype']  # Make sure to replace 'classifierColumn' with your actual target column name

    # Predict with the model and update the metric based on the prediction
    y_pred = model.predict_one(x)
    metric.update(y, y_pred)  # Correctly update the metric without reassignment

    # Learn from the true instance
    model.learn_one(x, y)
    total_trained_rows += 1

print(f"Final accuracy: {metric.get()}")

# Optionally, save the model
model_path = "hoeffding_tree_model.joblib"
dump(model, model_path)
print("Model saved.")
print(f"Model updated and saved. Total trained rows: {total_trained_rows}.")

Final accuracy: 0.27
Model saved.
Model updated and saved. Total trained rows: 200.


In [60]:
# Show model parameters

value_counts = df[classifierColumn].value_counts()
value_counts

fraudtype
Credit Card Fraud    50
Identity Theft       44
Insurance Fraud      42
Payment Fraud        37
Tax Evasion          27
Name: count, dtype: int64